##### Get Latest List of all Projects from FDOT website - run this code once after each cut off

In [1]:
import os
import time
import shutil
import glob
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

# Get Contracts Data till last month
df_last = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Combined_Till_Last_Month.xlsx")
print(f"Loaded {df_last.shape[0]} contracts from last month.")
df_latest = pd.read_excel(r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Latest.xlsx")
print(f"Loaded {df_latest.shape[0]} contracts from latest data.")
merged_df = pd.concat([df_last, df_latest], ignore_index=True)
print(f"Merged DataFrame shape: {merged_df.shape}")
merged_df = merged_df.drop_duplicates()
print(f"After dropping duplicates, DataFrame shape: {merged_df.shape}")
# Save the merged DataFrame to a new Excel file
output_file = r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Combined_Till_Last_Month.xlsx"
merged_df.to_excel(output_file, index=False)
print('Saved Combined COntract Data Till Last Month')
# Set download directory
root_dir=r"C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data"
# root_dir = input("Enter the folder path where All Contracts should be downloaded:  ")
os.makedirs(root_dir, exist_ok=True)

# Set up Chrome options
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": root_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True}
options.add_experimental_option("prefs", prefs)

# Start browser
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

# Open the URL
url = 'https://scoc.fdot.gov/#/active/1'
driver.get(url)

# Wait for overlay to disappear
try:
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, "div.overlay")))
except:
    pass

# Wait for the Excel export button
excel_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//span[contains(@class, "fa-file-excel")]')))

# Scroll into view and click
driver.execute_script("arguments[0].scrollIntoView(true);", excel_button)
time.sleep(1)
try:
    excel_button.click()
except:
    driver.execute_script("arguments[0].click();", excel_button)

# Wait for download to complete
time.sleep(30)  # Adjust if needed

# Get list of all Excel files in the download directory
excel_files = glob.glob(os.path.join(root_dir, "*.xlsx"))
#r'C:\Users\IlaBarshilia\OneDrive - OIA Global\VS_Code\'
if not excel_files:
    print("No Excel files found.")
    driver.quit()
    exit()

# Sort files by modification time (latest first)
latest_file = max(excel_files, key=os.path.getmtime)

today = datetime.today().strftime('%Y-%b-%d')
# Rename the latest file
desired_filename = "FDOT_All_Contracts_Latest.xlsx"
desired_filepath = os.path.join(root_dir, desired_filename)
print(desired_filepath)


if os.path.exists(desired_filepath):
    os.remove(desired_filepath)

os.rename(latest_file, desired_filepath)
# Clean up
driver.quit()
# print(f"Filtered data saved to: {output_file}")

df = pd.read_excel(desired_filepath)
# df = df[df["Active"] == True]
df.to_excel(desired_filepath, index=False)
print("Saved latest data!")

Loaded 4881 contracts from last month.
Loaded 1486 contracts from latest data.
Merged DataFrame shape: (6367, 30)
After dropping duplicates, DataFrame shape: (5645, 30)
Saved Combined COntract Data Till Last Month
C:\Users\IlaBarshilia\ACME Barricades\FDOT Web Scraping - FDOT Web Scraping Data\Source Data\FDOT_All_Contracts_Latest.xlsx
